## Submit a VeRL RL post-training job

This notebook prepares data and submits a VeRL GRPO training job on GSM8K to the Ray cluster.

**Run this notebook inside the Jupyter container** (the one you opened in your browser at `http://<IP>:8888`).

### Install dependencies and prepare data

In [ ]:
!pip install datasets

In [ ]:
import re, os, datasets

dataset = datasets.load_dataset("openai/gsm8k", "main")
train = dataset["train"].select(range(200))
test = dataset["test"].select(range(100))

def process(example, idx, split):
    q = example["question"]
    a = example["answer"]
    sol = re.search(r"#### (\-?[0-9\.\,]+)", a).group(1).replace(",","")
    return {"data_source":"openai/gsm8k","prompt":[{"role":"user","content":q+' Let\'s think step by step and output the final answer after "####".'}],"ability":"math","reward_model":{"style":"rule","ground_truth":sol},"extra_info":{"split":split,"index":idx,"answer":a,"question":q}}

train = train.map(lambda ex,i: process(ex,i,"train"), with_indices=True)
test = test.map(lambda ex,i: process(ex,i,"test"), with_indices=True)

os.makedirs("data/gsm8k", exist_ok=True)
train.to_parquet("data/gsm8k/train.parquet")
test.to_parquet("data/gsm8k/test.parquet")
print("Done!")

In [ ]:
!ls data/gsm8k/train.parquet data/gsm8k/test.parquet
!echo $RAY_ADDRESS

### Submit the VeRL GRPO training job

This submits an RL post-training job using:
- **GRPO** algorithm (no critic model needed)
- **Qwen2.5-0.5B-Instruct** (smallest model)
- **HF rollout** (compatible with MI100 GPUs)
- **1 epoch** on a small GSM8K subset (smoke test)

In [ ]:
!ray job submit --address="http://ray-head:8265" \
  --working-dir . \
  -- python3 -m verl.trainer.main_ppo \
  algorithm.adv_estimator=grpo \
  data.train_files=data/gsm8k/train.parquet \
  data.val_files=data/gsm8k/test.parquet \
  data.train_batch_size=8 \
  data.max_prompt_length=256 \
  data.max_response_length=128 \
  data.filter_overlong_prompts=True \
  data.truncation=error \
  actor_rollout_ref.model.path=Qwen/Qwen2.5-0.5B-Instruct \
  actor_rollout_ref.rollout.name=hf \
  actor_rollout_ref.rollout.tensor_model_parallel_size=1 \
  actor_rollout_ref.rollout.n=2 \
  actor_rollout_ref.rollout.top_k=0 \
  actor_rollout_ref.actor.optim.lr=1e-6 \
  actor_rollout_ref.actor.ppo_mini_batch_size=8 \
  actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=4 \
  actor_rollout_ref.actor.use_kl_loss=True \
  actor_rollout_ref.actor.kl_loss_coef=0.001 \
  actor_rollout_ref.model.enable_gradient_checkpointing=True \
  actor_rollout_ref.ref.log_prob_micro_batch_size_per_gpu=4 \
  algorithm.use_kl_in_reward=False \
  trainer.n_gpus_per_node=1 \
  trainer.nnodes=2 \
  trainer.total_epochs=1 \
  trainer.save_freq=-1 \
  trainer.test_freq=1 \
  trainer.val_before_train=False \
  'trainer.logger=["console"]'

### Monitor

Open the Ray dashboard in your browser at `http://<YOUR_IP>:8265` to watch the job progress.

The job should go: **PENDING → RUNNING → SUCCEEDED**